
# Delta Lake Tests

En este notebook se realizarán pruebas controladas sobre Delta Lake utilizando datos del proyecto.

El objetivo es practicar:

- Historial de versiones de tablas Delta.
- Transaction Log.
- Operaciones `UPDATE` y `DELETE`.
- Time Travel mediante `VERSION AS OF`.
- Recuperación y comparación de versiones.

Las pruebas se realizarán sobre una tabla independiente para evitar modificar las tablas Gold utilizadas en el análisis.

In [0]:
%sql

CREATE OR REPLACE TABLE olist.gold.order_analysis_delta_test AS
SELECT * FROM olist.gold.order_analysis LIMIT 10000

In [0]:
%sql
SELECT COUNT(*) FROM olist.gold.order_analysis_delta_test;

In [0]:
%sql

DESCRIBE HISTORY olist.gold.order_analysis_delta_test;

In [0]:
%sql

SELECT
    order_id,
    order_status,
    delay_days,
    is_late_delivery_business
FROM olist.gold.order_analysis_delta_test
LIMIT 1;

In [0]:
%sql
update olist.gold.order_analysis_delta_test
set delay_days=1
where order_id='b276e4f8c0fb86bd82fce576f21713e0'

In [0]:
%sql
select 
    order_id,
    order_status,
    delay_days,
    is_late_delivery_business
from olist.gold.order_analysis_delta_test
where order_id='b276e4f8c0fb86bd82fce576f21713e0'

In [0]:
%sql
describe history olist.gold.order_analysis_delta_test

In [0]:
%sql
SELECT
    order_id,
    order_status,
    delay_days,
    is_late_delivery_business
FROM olist.gold.order_analysis_delta_test VERSION AS OF 0
WHERE order_id = 'b276e4f8c0fb86bd82fce576f21713e0';

In [0]:
%sql
SELECT
    order_id,
    order_status,
    delay_days,
    is_late_delivery_business
FROM olist.gold.order_analysis_delta_test
WHERE order_id = 'b276e4f8c0fb86bd82fce576f21713e0';

In [0]:
%sql
DELETE FROM olist.gold.order_analysis_delta_test
where order_id='b276e4f8c0fb86bd82fce576f21713e0'

In [0]:
%sql
SELECT
    order_id,
    order_status,
    delay_days,
    is_late_delivery_business
FROM olist.gold.order_analysis_delta_test
WHERE order_id = 'b276e4f8c0fb86bd82fce576f21713e0';

In [0]:
%sql
DESCRIBE HISTORY olist.gold.order_analysis_delta_test

In [0]:
%sql
restore table olist.gold.order_analysis_delta_test to version as of 1

In [0]:
%sql
SELECT
    order_id,
    order_status,
    delay_days,
    is_late_delivery_business
FROM olist.gold.order_analysis_delta_test
WHERE order_id = 'b276e4f8c0fb86bd82fce576f21713e0';

In [0]:
%sql
describe history olist.gold.order_analysis_delta_test


# Resumen - Delta Lake Tests

En este notebook se realizaron pruebas controladas sobre Delta Lake utilizando una tabla independiente creada a partir de los datos del proyecto.

El objetivo fue practicar el funcionamiento del versionado de Delta Lake, el Transaction Log, Time Travel y la recuperación de estados anteriores de una tabla.

---

## 1. Creación de la tabla de prueba

Se creó una tabla independiente:

`olist.gold.order_analysis_delta_test`

La tabla fue generada a partir de `olist.gold.order_analysis` mediante una operación **CTAS (CREATE TABLE AS SELECT)**.

Se utilizaron **10.000 registros** para realizar las pruebas sin modificar las tablas Gold utilizadas por el análisis principal.

---

## 2. Historial inicial de Delta Lake

Se utilizó:

`DESCRIBE HISTORY`

para consultar el historial de operaciones de la tabla.

La creación inicial quedó registrada como:

- **Versión 0**
- Operación: `CREATE OR REPLACE TABLE AS SELECT`

Esto permitió comprobar que Delta Lake registra las operaciones realizadas sobre una tabla mediante su Transaction Log.

---

## 3. Prueba de UPDATE

Se seleccionó un `order_id` específico y se modificó de forma controlada uno de sus valores mediante `UPDATE`.

Posteriormente se verificó que el registro había sido actualizado correctamente.

Al consultar nuevamente el historial se generó:

- **Versión 1**
- Operación: `UPDATE`
- Filas actualizadas: **1**

El Transaction Log permitió identificar tanto la operación realizada como el registro afectado y las métricas asociadas.

---

## 4. Time Travel

Después del `UPDATE` se utilizó Time Travel mediante:

`VERSION AS OF`

Se comparó el mismo pedido entre:

- la versión original de la tabla;
- la versión actual después de la modificación.

Esto permitió comprobar que Delta Lake mantiene acceso al estado anterior de los datos aunque posteriormente hayan sido modificados.

---

## 5. Prueba de DELETE

Se realizó posteriormente un `DELETE` controlado sobre un `order_id`.

Se comprobó que el registro dejó de existir en la versión actual de la tabla.

El historial mostró una nueva operación:

- **Versión 2**
- Operación: `DELETE`
- Filas eliminadas: **1**

Mediante Time Travel se consultó nuevamente la versión anterior y se confirmó que el registro eliminado seguía disponible en el estado histórico de la tabla.

---

## 6. RESTORE

Finalmente se utilizó `RESTORE` para recuperar la tabla al estado correspondiente a la **versión 1**.

El historial registró:

- **Versión 3**
- Operación: `RESTORE`
- Versión restaurada: **1**

Esto permitió recuperar el estado existente después del `UPDATE` pero antes del `DELETE`.

Es importante destacar que `RESTORE` no elimina ni reemplaza el historial anterior. En su lugar, genera una **nueva versión de la tabla** cuyo contenido corresponde al estado seleccionado.

---

## Historial generado

Durante las pruebas se generó la siguiente secuencia:

| Versión | Operación | Descripción |
|---|---|---|
| 0 | CREATE OR REPLACE TABLE AS SELECT | Creación inicial de la tabla de prueba |
| 1 | UPDATE | Modificación controlada de un registro |
| 2 | DELETE | Eliminación controlada de un registro |
| 3 | RESTORE | Recuperación del estado correspondiente a la versión 1 |

---

## Funcionalidades de Delta Lake comprobadas

Durante este notebook se practicaron y verificaron:

- `CREATE TABLE AS SELECT`
- `UPDATE`
- `DELETE`
- `DESCRIBE HISTORY`
- Transaction Log
- Time Travel con `VERSION AS OF`
- `RESTORE`
- métricas de las operaciones Delta

---

## Conclusiones

Las pruebas permitieron comprobar que Delta Lake mantiene un historial de las modificaciones realizadas sobre las tablas.

Cada operación genera una nueva versión sin eliminar automáticamente los estados anteriores, lo que permite:

- realizar auditorías sobre cambios;
- consultar datos históricos;
- comparar diferentes versiones;
- recuperar información modificada o eliminada;
- restaurar una tabla a un estado anterior;
- mantener trazabilidad sobre las operaciones realizadas.

Estas características proporcionan mayor confiabilidad y control sobre los datos almacenados en el Lakehouse.

Con estas pruebas queda completada la validación de las principales funcionalidades de versionado y recuperación de Delta Lake utilizadas en el proyecto.